# 03 — Score → Validate → Pivot (Clean)

**Purpose**
Convert the manually scored long-format evaluation table (75 rows = 25 questions × 3 conditions)
into:
1) a cleaned canonical long table (still 75 rows), and
2) a validated wide table (25 rows; unit of analysis = question) for paired analyses.

**Input**
- `../data/interim/buddi_paper_labels_long.csv`

**Outputs**
- `../data/processed/buddi_paper_question_eval_long.csv`
- `../data/processed/buddi_paper_question_eval_wide.csv`
- `../outputs/buddi_paper/v1/logs/buddi_paper_v1_manifest_03.json`
- `../outputs/buddi_paper/v1/logs/data_dictionary_long.csv`
- `../outputs/buddi_paper/v1/logs/data_dictionary_wide.csv`
- `../outputs/buddi_paper/v1/logs/missingness_process_long.csv`

**Unit of analysis**
Question (N=25). Each question has exactly 3 rows (raw/struct/sem) in the long input.

**Reproducibility**
- Deterministic mappings for dataset labels, complexity, correctness.
- Fail-fast validation to prevent silent pairing errors.

In [ ]:
# Step 0 - Imports, paths and constraints and load
# --------------------------

import os, json, hashlib
import numpy as np
import pandas as pd
from datetime import datetime
from pathlib import Path

PAPER_SLUG = "buddi_paper"
RELEASE_ID = "v1"

IN_PATH = "../data/interim/buddi_paper_labels_long.csv"

OUT_LONG = f"../data/processed/{PAPER_SLUG}_question_eval_long.csv"
OUT_WIDE = f"../data/processed/{PAPER_SLUG}_question_eval_wide.csv"

LOG_DIR = f"../outputs/{PAPER_SLUG}/{RELEASE_ID}/logs"
Path("../data/processed").mkdir(parents=True, exist_ok=True)
Path(LOG_DIR).mkdir(parents=True, exist_ok=True)

NOW = datetime.now().isoformat(timespec="seconds")

df_raw = pd.read_csv(IN_PATH)
print("Loaded:", IN_PATH)
print("Shape:", df_raw.shape)
display(df_raw.head(3))

Loaded: ../data/interim/buddi_paper_labels_long.csv
Shape: (75, 14)


,Questionid,Datum,Dataset,Question Theme,Complexity Level,Correct2,Token count,Time to first token (ms),Structural assumption,Correct datasource/column,Semantic assumption,Correct interpretation field,Error Reason,Explanation error
0,1,30-Oct,Semantic,Development,General,Yes,13601,1816,No,NaN,No,NaN,NaN,NaN
1,2,30-Oct,Semantic,Development,Direct Factual,Yes,21292,630,No,Yes,No,Yes,NaN,NaN
2,3,30-Oct,Semantic,Development,Direct Analytical,Yes,36170,1086,No,Yes,No,Yes,NaN,NaN


## Step 1 — Normalize column names

We normalize column names into stable snake_case names used by analysis notebooks.
This avoids accidental breakage if spreadsheet headers change formatting.

In [ ]:
df = df_raw.copy()

# Update this mapping only if input headers change.
rename_map = {
    "Questionid": "question_id",
    "Datum": "date",
    "Dataset": "dataset",
    "Question Theme": "question_theme",
    "Complexity Level": "complexity_label",
    "Correct2": "correct",
    "Token count": "tokens",
    "Time to first token (ms)": "ttft_ms",
    "Structural assumption": "structural_assumption_made",
    "Correct datasource/column": "source_correct",
    "Semantic assumption": "semantic_assumption_made",
    "Correct interpretation field": "interpretation_correct",
    "Error Reason": "error_reason",
}

df = df.rename(columns=rename_map)

required = [
    "question_id", "dataset", "question_theme", "complexity_label", "correct",
    "tokens", "ttft_ms", "structural_assumption_made", "semantic_assumption_made",
    "source_correct", "interpretation_correct", "error_reason"
]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns after rename: {missing}")

print("Columns OK.")


Columns OK.


## Step 2 — Normalize key fields (dataset, complexity, correctness)

We map:
- Dataset → {raw, struct, sem}
- Complexity label → {1..5}
- Correctness → {0,1}

We fail loudly on unexpected values.

In [ ]:
# Dataset normalization
ds_map = {"raw": "raw", "structural": "struct", "semantic": "sem"}
df["dataset_norm"] = df["dataset"].astype(str).str.strip().str.lower().map(ds_map)
bad_ds = df.loc[df["dataset_norm"].isna(), "dataset"].unique()
if len(bad_ds) > 0:
    raise ValueError(f"Unexpected Dataset values: {bad_ds}")

# Complexity label -> numeric
comp_map = {
    "general": 1,
    "direct factual": 2,
    "direct analytical": 3,
    "multi-step analytical": 4,
    "advanced analytical": 5,
}
df["complexity_level"] = df["complexity_label"].astype(str).str.strip().str.lower().map(comp_map)
bad_comp = df.loc[df["complexity_level"].isna(), "complexity_label"].unique()
if len(bad_comp) > 0:
    raise ValueError(f"Unexpected Complexity Level labels: {bad_comp}")

# Correctness Yes/No -> 1/0
corr_map = {"yes": 1, "no": 0}
df["correct01"] = df["correct"].astype(str).str.strip().str.lower().map(corr_map)
bad_corr = df.loc[df["correct01"].isna(), "correct"].unique()
if len(bad_corr) > 0:
    raise ValueError(f"Unexpected Correct2 values: {bad_corr}")

# Numeric conversions
df["tokens"] = pd.to_numeric(df["tokens"], errors="coerce")
df["ttft_ms"] = pd.to_numeric(df["ttft_ms"], errors="coerce")

if df["tokens"].isna().any():
    bad_rows = df[df["tokens"].isna()][["question_id","dataset","tokens"]].head(10)
    raise ValueError(f"Non-numeric tokens detected. Example rows:\n{bad_rows}")

print("Normalization OK.")
print("Dataset counts:\n", df["dataset_norm"].value_counts())
print("Complexity counts (long; should be 15 per level):\n", df["complexity_level"].value_counts().sort_index())


Normalization OK.
Dataset counts:
 dataset_norm
sem       25
struct    25
raw       25
Name: count, dtype: int64
Complexity counts (long; should be 15 per level):
 complexity_level
1    15
2    15
3    15
4    15
5    15
Name: count, dtype: int64


## Step 3 — Derive `error_category`

Rule:
- If correct01==1 → error_category = "none"
- Else map known error reasons to {structural, semantic, computational}, preserving unknown strings for audit.

In [ ]:
er = df["error_reason"].astype(str).str.strip().str.lower()
er = er.replace({"n/a": np.nan, "na": np.nan, "nan": np.nan, "": np.nan})

er_map = {
    "structural": "structural",
    "semantic": "semantic",
    "computational/agentic": "computational",
    "agentic": "computational",
    "computational": "computational", 
    "computational / agentic": "computational",

}

mapped = er.map(er_map)

# correct answers always "none"
df["error_category"] = np.where(df["correct01"] == 1, "none", mapped)

# incorrect answers must map to one of the allowed categories
bad = df[(df["correct01"] == 0) & (df["error_category"].isna())]
if len(bad) > 0:
    ex = bad[["Questionid","Dataset","error_reason","correct01"]].head(20)
    raise ValueError(
        "Found incorrect answers with unmapped/unknown error_reason values. "
        "Update er_map to cover them. Examples:\n" + ex.to_string(index=False)
    )

# incorrect answers must not have 'none'
bad_none = df[(df["correct01"] == 0) & (df["error_category"] == "none")]
if len(bad_none) > 0:
    ex = bad_none[["Questionid","Dataset","error_reason","correct01"]].head(20)
    raise ValueError(
        "Found incorrect answers with error_category == 'none'. Fix the input labels. "
        "Examples:\n" + ex.to_string(index=False)
    )

display(df[["correct01","error_reason","error_category"]].head(10))


,correct01,error_reason,error_category
0,1,NaN,none
1,1,NaN,none
2,1,NaN,none
3,1,NaN,none
4,1,NaN,none
5,1,NaN,none
6,1,NaN,none
7,1,NaN,none
8,1,NaN,none
9,0,Structural,structural


## Step 4 — Validate long table integrity (must pass)

Checks:
- Exactly 75 rows (25×3)
- Exactly 25 unique question_id
- Exactly 3 rows per question_id
- Each question_id includes all three dataset_norm values
- Complexity consistent within each question_id

In [ ]:
if len(df) != 75:
    raise ValueError(f"Expected 75 rows (25×3); found {len(df)}")
if df["question_id"].nunique() != 25:
    raise ValueError(f"Expected 25 unique question_id; got {df['question_id'].nunique()}")

vc = df["question_id"].value_counts()
if not (vc == 3).all():
    raise ValueError(f"Each question_id must appear exactly 3 times. Offenders:\n{vc[vc!=3]}")

has_all = df.groupby("question_id")["dataset_norm"].nunique()
if not (has_all == 3).all():
    raise ValueError(f"Some questions missing a condition. Offenders:\n{has_all[has_all!=3]}")

comp_consistent = df.groupby("question_id")["complexity_level"].nunique()
if not (comp_consistent == 1).all():
    raise ValueError(f"Complexity differs within question_id. Offenders:\n{comp_consistent[comp_consistent!=1]}")

print("Long table validations passed.")


Long table validations passed.


## Step 5 — Export audit artifacts

We export:
- a data dictionary (types, missingness, unique values preview)
- process-variable missingness summary

Useful for internal review and supplementary reporting.

In [ ]:
def make_data_dictionary(df_in: pd.DataFrame) -> pd.DataFrame:
    rows = []
    n = len(df_in)
    for col in df_in.columns:
        s = df_in[col]
        miss = int(s.isna().sum())
        uniq = s.dropna().unique()
        uniq_preview = ", ".join(map(str, uniq[:10])) + (" ..." if len(uniq) > 10 else "")
        rows.append({
            "column": col,
            "dtype": str(s.dtype),
            "missing_n": miss,
            "missing_pct": miss / n if n else np.nan,
            "n_unique_nonmissing": int(len(uniq)),
            "unique_preview": uniq_preview
        })
    return pd.DataFrame(rows).sort_values("column").reset_index(drop=True)

dd_long = make_data_dictionary(df)
dd_long.to_csv(f"{LOG_DIR}/data_dictionary_long.csv", index=False)

process_cols = ["structural_assumption_made","semantic_assumption_made","source_correct","interpretation_correct"]
miss = df[process_cols].isna().sum().to_frame("missing_n")
miss["missing_pct"] = miss["missing_n"] / len(df)
miss.to_csv(f"{LOG_DIR}/missingness_process_long.csv")

print("Wrote:")
print(" -", f"{LOG_DIR}/data_dictionary_long.csv")
print(" -", f"{LOG_DIR}/missingness_process_long.csv")


Wrote:
 - ../outputs/buddi_paper/v1/logs/data_dictionary_long.csv
 - ../outputs/buddi_paper/v1/logs/missingness_process_long.csv


## Step 6 — Pivot to wide (25 rows)

The wide table is the analysis-ready format:
- one row per question
- columns suffixed by _raw/_struct/_sem

In [ ]:
index_cols = ["question_id", "question_theme", "complexity_level"]

def pivot_field(field: str, prefix: str):
    p = df.pivot_table(index=index_cols, columns="dataset_norm", values=field, aggfunc="first")
    p.columns = [f"{prefix}_{c}" for c in p.columns]
    return p

wide = pd.concat([
    pivot_field("correct01", "correct"),
    pivot_field("tokens", "tokens"),
    pivot_field("ttft_ms", "TTFT"),
    pivot_field("error_category", "error_category"),
    pivot_field("structural_assumption_made", "structural_assumption_made"),
    pivot_field("semantic_assumption_made", "semantic_assumption_made"),
    pivot_field("source_correct", "source_correct"),
    pivot_field("interpretation_correct", "interpretation_correct"),
], axis=1).reset_index()

# Validate wide
if len(wide) != 25 or wide["question_id"].nunique() != 25:
    raise ValueError("Wide pivot failed: expected 25 unique question rows.")

cnt = wide["complexity_level"].value_counts().sort_index()
for lvl in [1,2,3,4,5]:
    if int(cnt.get(lvl,0)) != 5:
        raise ValueError(f"Expected 5 questions per complexity level. Got: {cnt.to_dict()}")

for c in ["raw","struct","sem"]:
    vals = set(wide[f"correct_{c}"].dropna().unique())
    if not vals.issubset({0,1}):
        raise ValueError(f"correct_{c} is not binary. Values: {vals}")

print("Wide table validations passed.")
display(wide.head(3))


Wide table validations passed.


,question_id,question_theme,complexity_level,correct_raw,correct_sem,correct_struct,tokens_raw,tokens_sem,tokens_struct,TTFT_raw,...,structural_assumption_made_struct,semantic_assumption_made_raw,semantic_assumption_made_sem,semantic_assumption_made_struct,source_correct_raw,source_correct_sem,source_correct_struct,interpretation_correct_raw,interpretation_correct_sem,interpretation_correct_struct
0,1,Development,1,1,1,1,80293,13601,12597,2205,...,No,No,No,No,NaN,NaN,NaN,NaN,NaN,NaN
1,2,Development,2,0,1,0,243787,21292,21017,1831,...,No,Yes,No,Yes,Yes,Yes,Yes,No,Yes,No
2,3,Development,3,0,1,0,80829,36170,33629,2142,...,Yes,Yes,No,Yes,Yes,Yes,No,NaN,Yes,NaN


In [ ]:
wide.to_csv(OUT_WIDE, index=False)
df.to_csv(OUT_LONG, index=False)

dd_wide = make_data_dictionary(wide)
dd_wide.to_csv(f"{LOG_DIR}/data_dictionary_wide.csv", index=False)

def sha256_file(path: str) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

manifest = {
    "paper_slug": PAPER_SLUG,
    "release_id": RELEASE_ID,
    "created_at": NOW,
    "input_path": IN_PATH,
    "input_sha256": sha256_file(IN_PATH),
    "outputs": {
        "clean_long": OUT_LONG,
        "wide": OUT_WIDE,
        "data_dictionary_long": f"{LOG_DIR}/data_dictionary_long.csv",
        "data_dictionary_wide": f"{LOG_DIR}/data_dictionary_wide.csv",
        "missingness_process_long": f"{LOG_DIR}/missingness_process_long.csv",
    },
    "row_counts": {
        "input_rows": int(len(df_raw)),
        "clean_long_rows": int(len(df)),
        "wide_rows": int(len(wide)),
    }
}

with open(f"{LOG_DIR}/{PAPER_SLUG}_{RELEASE_ID}_manifest_03.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print("Wrote:")
print(" -", OUT_LONG)
print(" -", OUT_WIDE)
print(" -", f"{LOG_DIR}/data_dictionary_wide.csv")
print(" -", f"{LOG_DIR}/{PAPER_SLUG}_{RELEASE_ID}_manifest_03.json")


Wrote:
 - ../data/processed/buddi_paper_question_eval_long.csv
 - ../data/processed/buddi_paper_question_eval_wide.csv
 - ../outputs/buddi_paper/v1/logs/data_dictionary_wide.csv
 - ../outputs/buddi_paper/v1/logs/buddi_paper_v1_manifest_03.json
